# Converting coordinates from a csv file to a json file

In [1]:
import json
import os
import pandas as pd
import numpy as np
import rasterio

from PIL import Image
from pyproj import Transformer   
from datetime import datetime    

## Import the raster file to get the metadata

We need the information from the raster as the json file needs to be in reference to the origin and transformation of the tif file.

In [2]:
tif_base_dir = 'testing/raw_images/'
tif_name = '20240101_mimal_test'
tif_path = os.path.join(tif_base_dir, tif_name + '.tif')

with rasterio.open(tif_path) as raster:
    # Read the raster band
    imported_raster = raster.read(1)
    # Get the metadata of the raster
    imported_raster_meta = raster.meta
    # Get the raster transform parameters
    raster_transform = raster.transform

print("Shape of the raster (rows, columns):")
print(imported_raster.shape)
print("\n")

print("Raster metadata:")
print(imported_raster_meta)
print("\n")

print("Affine transformation parameters:")
print(raster_transform)

Shape of the raster (rows, columns):
(11309, 14180)


Raster metadata:
{'driver': 'GTiff', 'dtype': 'uint16', 'nodata': 0.0, 'width': 14180, 'height': 11309, 'count': 4, 'crs': CRS.from_epsg(32753), 'transform': Affine(3.0, 0.0, 438081.0,
       0.0, -3.0, 8533512.0)}


Affine transformation parameters:
| 3.00, 0.00, 438081.00|
| 0.00,-3.00, 8533512.00|
| 0.00, 0.00, 1.00|


## Account for the png padding

As the png has padding, we need to figure out how much to add to the x and y coordinates to get the correct position for the labels in the png file.

This function will be different if there is padding on both sides and the top and bottom of the image. I've just accounted for padding on the left and top of the image by adding the difference in between the png size and the raster size to the x (difference in width) and y (difference in height) coordinates.

If the padding is consistent (e.g. tile_size - stride), we can just add that padding to the x and y coordinates. In that case, set width_diff and height_diff to the padding amount for the x and y axes respectively.

In [3]:
Image.MAX_IMAGE_PIXELS = 933120000 # change to greater than the nnumber of pixels (only needed if there's a warning)

# Base directory for the png image
png_base_dir = 'testing/pngs/'

# Load the png image
image = Image.open(f'{png_base_dir}20240101_mimal_test.png')

# Convert the image to a numpy array
pixel_data = np.array(image)

## Get image information

In [5]:
# Get image information
png_width, png_height = image.size
print(f'Image size: ({png_width}, {png_height})')

# Calculate the different in the width and height of the image and the raster
width_diff = png_width - imported_raster.shape[1]
height_diff = png_height - imported_raster.shape[0]
print(f'Difference in width: {width_diff}')
print(f'Difference in height: {height_diff}')


# if using images with padded left and right (and top and bottom)
tile_size = 416
stride = 104
min_pad = tile_size - stride
# subtract the padding on the right and bottom from the differences
width_diff = width_diff - min_pad
height_diff = height_diff - min_pad
print(f'Difference in width after removing right padding: {width_diff}')
print(f'Difference in height after removing bottom padding: {height_diff}')

Image size: (14976, 12064)
Difference in width: 796
Difference in height: 755
Difference in width after removing right padding: 484
Difference in height after removing bottom padding: 443


## Convert the locations in the csv file to the same projection as the tif file

As the locations are in WGS84, we need to convert them to the same projection as the tif file, and then convert them to pixel coordinates using the raster transformation from the metadata.

In [6]:
# Create the reprojection function
coord_transformer = Transformer.from_crs('epsg:4326', 'epsg:32753', always_xy=True)

# Test on sample set of coordinates
x,y = coord_transformer.transform(133.6153775, -13.7976017)
print(x,y)

# Check the pixel coordinates of the transformed coordinates
pixel_column, pixel_row = ~raster_transform * (x, y)
print(pixel_column, pixel_row)

350330.60260668746 8474226.385882989
-29250.132464437513 19761.871372337453


## Read in the csv file of labelled coordinates

In [7]:
# Specify the path to your CSV file
csv_base_dir = f'data/'
# csv_name = '2024 Rapid Waterhole Assessment_aligned.csv' # with health states
csv_name = '2024 Rapid Waterhole Assessment_aligned.csv'
csv_file_path = os.path.join(csv_base_dir, csv_name)

# Read the CSV file into a DataFrame
waterhole_labelled_df = pd.read_csv(csv_file_path)

print(f'Number of samples: {len(waterhole_labelled_df)}')
print('\n')

# Display the first few rows of the DataFrame
print(waterhole_labelled_df.head())

Number of samples: 122


   OBJECTID   Timestamp   Latitude   Longitude  Class        Observer  \
0         1  2024-05-07 -13.657141  134.354081      2  Andrew Hoskins   
1         2  2024-05-07 -13.655816  134.371830      2  Andrew Hoskins   
2         3  2024-05-07 -13.655044  134.393416      3  Andrew Hoskins   
3         4  2024-05-07 -13.659928  134.417495      2  Andrew Hoskins   
4         5  2024-05-07 -13.657286  134.525346      2  Andrew Hoskins   

   ObserverSi Water_Type_Chopper Water_Type_Satellite transectNu permanence  \
0  Back Right                NaN                River        m08          P   
1  Back Right                NaN      River/Billabong        m08          P   
2  Back Right                NaN            Billabong        m08     I or E   
3  Back Right                NaN               Stream        m08     I or E   
4  Back Right                NaN            Billabong        m08     I or E   

     ID Wet_Dry_Chopper Wet_Dry_Satellite  
0   NaN          

## Reproject the label coordinates to the same projection as the tif file

In [8]:
x_proj, y_proj = coord_transformer.transform(
    waterhole_labelled_df['Longitude'].values, 
    waterhole_labelled_df['Latitude'].values
)

print(x_proj, y_proj)

[430143.17685067 432062.43965287 434396.87455498 437002.42767336
 448666.15966412 458053.23977042 458237.10489792 458987.86848127
 461874.23246752 449507.09670538 445638.36590115 444025.54981718
 440794.70240962 435105.66625879 428903.23024269 425354.55770888
 424196.77749568 423212.16039536 421165.12538497 417110.28161243
 401196.41106128 391625.28131144 373867.42304861 371360.14238773
 358132.66349237 425752.81788484 426812.24890471 429614.87363284
 458855.26207083 449006.87935519 421700.49235191 421251.19090197
 420799.34395841 389044.36386998 406364.40849872 427547.09390463
 436041.47970019 445150.68701575 459923.74064893 514935.59402967
 382948.88660257 381597.26513111 386069.49127455 388823.87049276
 390486.87336337 379662.95731606 446908.44526433 451882.41080017
 453221.3678156  453138.96858241 455277.34374629 463802.47700669
 444476.01486059 369842.88634585 370015.66197925 371192.01958601
 394621.02910232 397881.89751955 367844.88179619 366973.09283339
 366804.6126904  366446.8

## Convert the label coordinates to pixel coordinates

These pixel coordinates are in reference to the tif file, not the png yet (if it has padding).

In [9]:
# Convert to pixel coordinates
pixel_coords = np.array([~raster_transform * (x, y) for x, y in zip(x_proj, y_proj)])
print(pixel_coords)

[[ -2645.94104978  14471.10079774]
 [ -2006.18678238  14420.5911999 ]
 [ -1228.04181501  14390.12407193]
 [  -359.52410888  14568.05094087]
 [  3528.38655471  14462.20206133]
 [  6657.41325681  14518.42543212]
 [  6718.70163264  14493.14788764]
 [  6968.95616042  14613.4865288 ]
 [  7931.07748917  14430.70499074]
 [  3808.69890179  10670.483244  ]
 [  2519.12196705  10092.17430934]
 [  1981.51660573  10642.84589428]
 [   904.56746987  10696.52231285]
 [  -991.77791374  10655.48777388]
 [ -3059.25658577  10707.92841225]
 [ -4242.14743037  10726.13536447]
 [ -4628.07416811  10725.38972459]
 [ -4956.27986821  10729.28584537]
 [ -5638.62487168  10474.58410868]
 [ -6990.23946252  10818.37665637]
 [-12294.86297957  12069.48225676]
 [-15485.23956285  11520.8599326 ]
 [-21404.52565046   9914.85006453]
 [-22240.28587076   9420.73343562]
 [-26649.44550254  12061.96713864]
 [ -4109.39403839  14318.03224081]
 [ -3756.2503651   14255.47342981]
 [ -2822.04212239  14418.08043699]
 [  6924.75402361  1

## Add the new columns to the dataframe

Here is where we add to the x and y coordinates to account for the padding in the png file.

In [10]:
# Add the new columns to the dataframe
waterhole_labelled_df['x_proj'] = x_proj
waterhole_labelled_df['y_proj'] = y_proj
waterhole_labelled_df['pixel_col'] = pixel_coords[:, 0] + width_diff
waterhole_labelled_df['pixel_row'] = pixel_coords[:, 1] + height_diff

print(waterhole_labelled_df.head())

   OBJECTID   Timestamp   Latitude   Longitude  Class        Observer  \
0         1  2024-05-07 -13.657141  134.354081      2  Andrew Hoskins   
1         2  2024-05-07 -13.655816  134.371830      2  Andrew Hoskins   
2         3  2024-05-07 -13.655044  134.393416      3  Andrew Hoskins   
3         4  2024-05-07 -13.659928  134.417495      2  Andrew Hoskins   
4         5  2024-05-07 -13.657286  134.525346      2  Andrew Hoskins   

   ObserverSi Water_Type_Chopper Water_Type_Satellite transectNu permanence  \
0  Back Right                NaN                River        m08          P   
1  Back Right                NaN      River/Billabong        m08          P   
2  Back Right                NaN            Billabong        m08     I or E   
3  Back Right                NaN               Stream        m08     I or E   
4  Back Right                NaN            Billabong        m08     I or E   

     ID Wet_Dry_Chopper Wet_Dry_Satellite         x_proj        y_proj  \
0   NaN     

## Define the function to create the json file

Essentially we are just taking the pixel coordinates and the labels and creating a json file with the correct format.

In [11]:
def csv_to_labelme(x, y, 
                   labels=None, 
                   image_path=None, 
                   image_height=None, 
                   image_width=None):
    """
    Convert coordinate columns from a DataFrame to LabelMe JSON format.
    
    Args:
        x (pd.Series): Series containing x/longitude coordinates
        y (pd.Series): Series containing y/latitude coordinates
        labels (pd.Series, optional): Series containing point labels. Defaults to None
        image_path (str, optional): Path to the corresponding image file. Defaults to None
        image_height (int, optional): Height of the image in pixels. Defaults to None
        image_width (int, optional): Width of the image in pixels. Defaults to None
        
    Returns:
        dict: LabelMe formatted JSON
    """
    # Validate inputs
    if len(x) != len(y):
        raise ValueError("x and y coordinates must have the same length")
    if labels is not None and len(labels) != len(x):
        raise ValueError("labels must have the same length as coordinates")
    
    # Initialize LabelMe JSON structure
    labelme_json = {
        "version": "5.0.1",
        "flags": {},
        "shapes": [],
        "imagePath": os.path.basename(image_path) if image_path else "",
        "imageData": None,  # LabelMe stores base64 image data here, but we'll leave it empty
        "imageHeight": image_height,
        "imageWidth": image_width
    }
    
    # Convert each point to LabelMe shape
    point_size = 5  # Size of the point representation in pixels
    
    for i in range(len(x)):
        # Skip if coordinates are NaN
        if pd.isna(x[i]) or pd.isna(y[i]):
            continue
            
        shape = {
            "label": str(labels.iloc[i]) if labels is not None else "point",
            "points": [
                [float(x[i]) - point_size, float(y[i]) - point_size],  # Top-left
                [float(x[i]) + point_size, float(y[i]) + point_size]   # Bottom-right
            ],
            "group_id": None,
            "shape_type": "rectangle",
            "flags": {}
        }
        
        labelme_json['shapes'].append(shape)
    
    # Add creation time
    labelme_json['timeStamp'] = datetime.now().isoformat()
    
    return labelme_json

## Function to save the json file

In [12]:
# Convert the DataFrame to LabelMe JSON
def save_labelme_json(labelme_json, output_path):
    """Save the LabelMe JSON to file."""
    with open(output_path, 'w') as f:
        json.dump(labelme_json, f, indent=2)

## Run the csv to json function

The image path should be the name of the png file, and the output path of the save_labelme_json function should lead to where the png is saved.

In [14]:
# Convert and save
labelme_json = csv_to_labelme(
    x=waterhole_labelled_df['pixel_col'],
    y=waterhole_labelled_df['pixel_row'],
    labels=waterhole_labelled_df['Class'],
    image_path=tif_name + '.png',
    image_height=png_height,
    image_width=png_width
)
    
# Save the LabelMe JSON file
save_labelme_json(labelme_json, f'{png_base_dir}/{tif_name}.json')
print(labelme_json)

{'version': '5.0.1', 'flags': {}, 'shapes': [{'label': '2', 'points': [[-2166.9410497775243, 14909.100797740743], [-2156.9410497775243, 14919.100797740743]], 'group_id': None, 'shape_type': 'rectangle', 'flags': {}}, {'label': '2', 'points': [[-1527.1867823759094, 14858.591199895367], [-1517.1867823759094, 14868.591199895367]], 'group_id': None, 'shape_type': 'rectangle', 'flags': {}}, {'label': '3', 'points': [[-749.0418150051846, 14828.124071931466], [-739.0418150051846, 14838.124071931466]], 'group_id': None, 'shape_type': 'rectangle', 'flags': {}}, {'label': '2', 'points': [[119.47589112032438, 15006.050940865185], [129.47589112032438, 15016.050940865185]], 'group_id': None, 'shape_type': 'rectangle', 'flags': {}}, {'label': '2', 'points': [[4007.3865547062014, 14900.202061329037], [4017.3865547062014, 14910.202061329037]], 'group_id': None, 'shape_type': 'rectangle', 'flags': {}}, {'label': '3', 'points': [[7136.4132568080095, 14956.425432116725], [7146.4132568080095, 14966.425432